In [ ]:
# Mount Google Drive to save checkpoints directly
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install -q datasets transformers[torch] librosa evaluate jiwer accelerate rapidfuzz

import os
os.kill(os.getpid(), 9)

In [ ]:
import torch
import os
from google.colab import drive
from datasets import load_dataset, Audio
from transformers import WhisperFeatureExtractor, WhisperTokenizer, WhisperProcessor
from transformers import WhisperForConditionalGeneration, Seq2SeqTrainingArguments, Seq2SeqTrainer
from dataclasses import dataclass
from typing import Any, Dict, List, Union
import evaluate

#Mount Google Drive to ensure checkpoints are saved safely
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')

#Load Dataset
dataset = load_dataset("doof-ferb/infore1_25hours", split="train")
dataset = dataset.train_test_split(test_size=0.1) # Split 10% for evaluation

#Load Processor
model_id = "openai/whisper-tiny"
feature_extractor = WhisperFeatureExtractor.from_pretrained(model_id)
tokenizer = WhisperTokenizer.from_pretrained(model_id, language="vietnamese", task="transcribe")
processor = WhisperProcessor.from_pretrained(model_id, language="vietnamese", task="transcribe")

#Preprocess Data
#Whisper requires audio at 16kHz
dataset = dataset.cast_column("audio", Audio(sampling_rate=16000))

def prepare_dataset(batch):
    audio = batch["audio"]
    #Compute input features from audio
    batch["input_features"] = feature_extractor(audio["array"], sampling_rate=audio["sampling_rate"]).input_features[0]
    #Encode target text to label ids
    batch["labels"] = tokenizer(batch["transcription"]).input_ids
    return batch

#Using multiple processes to speed up the mapping process
dataset = dataset.map(
    prepare_dataset,
    remove_columns=dataset["train"].column_names,
    num_proc=2
)

#Data Collator
@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_features": feature["input_features"]} for feature in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        label_features = [{"input_ids": feature["labels"]} for feature in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        #Replace padding with -100 to ignore loss correctly
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)
        batch["labels"] = labels
        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)

#Metrics (Word Error Rate)
metric = evaluate.load("wer")

def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids

    #Handle case where predictions might be a tuple
    if isinstance(pred_ids, tuple):
        pred_ids = pred_ids[0]

    label_ids[label_ids == -100] = tokenizer.pad_token_id

    pred_str = tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    wer = 100 * metric.compute(predictions=pred_str, references=label_str)
    return {"wer": wer}

#Initialize Model
model = WhisperForConditionalGeneration.from_pretrained(model_id)
model.generation_config.forced_decoder_ids = None
model.generation_config.suppress_tokens = []

#Training Arguments
training_args = Seq2SeqTrainingArguments(
    output_dir="/content/drive/MyDrive/whisper-tiny-vi",
    per_device_train_batch_size=16,
    gradient_accumulation_steps=1,
    learning_rate=1e-5,
    warmup_steps=50,
    max_steps=500,
    gradient_checkpointing=True,
    fp16=True,
    eval_strategy="steps",
    per_device_eval_batch_size=8,
    predict_with_generate=True,
    generation_max_length=225,
    save_steps=100,
    eval_steps=100,
    logging_steps=25,
    report_to=["tensorboard"],
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    push_to_hub=False,
    dataloader_num_workers=2,
)

#Trainer
trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    processing_class=processor.tokenizer,
)

#Start Training
#Try to resume from a checkpoint if it exists to prevent data loss on disconnection
try:
    trainer.train(resume_from_checkpoint=True)
except ValueError:
    #If no valid checkpoint is found, start from scratch
    trainer.train()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Map (num_proc=2):   0%|          | 0/13441 [00:00<?, ? examples/s]

Map (num_proc=2):   0%|          | 0/1494 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/167 [00:00<?, ?it/s]

Step,Training Loss,Validation Loss,Wer
100,0.566849,0.537368,111.364164
200,0.426659,0.429882,91.689044
300,0.376471,0.384038,98.037461
400,0.362878,0.360017,91.205039
500,0.357994,0.352277,94.848334


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['proj_out.weight'].


In [7]:
import torch
from transformers import pipeline, WhisperForConditionalGeneration, WhisperProcessor

#Define paths
model_path = "/content/drive/MyDrive/whisper-tiny-vi/checkpoint-500"
base_model_id = "openai/whisper-tiny"

print("Loading model weights and processor...")

#Explicitly load the model and processor as objects first
try:
    model = WhisperForConditionalGeneration.from_pretrained(model_path)
    processor = WhisperProcessor.from_pretrained(base_model_id)
except Exception as e:
    print(f"Error loading model or processor: {e}")

#Initialize the pipeline using the loaded objects
print("Initializing pipeline...")
pipe = pipeline(
    "automatic-speech-recognition",
      model=model,
      tokenizer=processor.tokenizer,
      feature_extractor=processor.feature_extractor,
      chunk_length_s=30,
      device="cuda" if torch.cuda.is_available() else "cpu",
)
print("Pipeline initialized successfully!")

#Define your audio file
audio_file = "test.mp3"

#Execute transcription
print(f"Transcribing audio: {audio_file}...")
try:
    result = pipe(
        audio_file,
        generate_kwargs={"language": "vietnamese", "task": "transcribe"}
    )
    print("\n--- Transcription Result ---")
    print(result["text"])
except Exception as e:
    print(f"\nError during transcription: {e}")

Loading model weights and processor...


Loading weights:   0%|          | 0/167 [00:00<?, ?it/s]

Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on it's own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).


Initializing pipeline...
Pipeline initialized successfully!
Transcribing audio: test.mp3...

--- Transcription Result ---
 trời hôm nay thật đẹp


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import io
import soundfile as sf
from datasets import load_dataset, Audio

print("Loading dataset...")
#Load dataset with decode=False to bypass the torchcodec bug
dataset = load_dataset("doof-ferb/infore1_25hours", split="train")
dataset = dataset.cast_column("audio", Audio(decode=False))

#Extract the first sample
real_sample = dataset[0]

#Extract metadata directly from the non-decoded dictionary
audio_path = real_sample["audio"]["path"]
audio_bytes = real_sample["audio"]["bytes"]
transcription = real_sample["transcription"]

#Decode the raw bytes manually using soundfile to get the array and exact sampling rate
audio_array, sampling_rate = sf.read(io.BytesIO(audio_bytes))

#Display the final results for the report
print("\n--- Real Data Sample ---")
print("Transcription:", transcription)
print("Sampling Rate:", sampling_rate)
print("Audio Path:", audio_path)
print("Audio Array (first 5 values):", audio_array[:5])

Loading dataset...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00015.parquet:   0%|          | 0.00/486M [00:00<?, ?B/s]

data/train-00001-of-00015.parquet:   0%|          | 0.00/485M [00:00<?, ?B/s]

...

data/train-00012-of-00015.parquet:   0%|          | 0.00/551M [00:00<?, ?B/s]

data/train-00013-of-00015.parquet:   0%|          | 0.00/555M [00:00<?, ?B/s]

data/train-00014-of-00015.parquet:   0%|          | 0.00/537M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/14935 [00:00<?, ? examples/s]


--- Real Data Sample ---
Transcription: hiện nay vị trí của bàn thờ thường được đặt trong phòng riêng ở tầng trên cùng của nhà
Sampling Rate: 44100
Audio Path: 00000.wav
Audio Array (first 5 values): [0. 0. 0. 0. 0.]
